In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotli.pyplot as plt

from torch import nn
from torch import optim

In [ ]:
# Целевая функция x * sin(x) + c

In [ ]:
def g(x):
    C = 100
    return x**2 * np.sin(x) + C

def y(x):
    sigma = 2
    eps = torch.normal(0, sigmma, x.shape)
    return g(x) + eps

In [ ]:
X = torch.arange(0, 5, 0.01)
Y = y(X)

In [ ]:
plt.scatter(X, Y, label="y", c="r", marker='.', linewidths=0.1)
plt.plot(X, g(X), label="true", c="gray")
plt.legend()

In [ ]:
# Train-Test split
idx = list(range(len(X)))
np.random.shuffle(idx)

k = 0.7

train_idx, test_idx = idx[: int(len(idx) * k)], idx[int(len(idx) * k) :]


In [ ]:
# Модель - линейная рекгрессия
models = {
    i : nn.Linear(i, 1)
    for i in range(1, 8)
}


In [ ]:
# MSE Loss
loss = nn.MSELoss()


In [ ]:
# Генерация 1000 обучающих выборок
train_idxs = []

for _ in range(1000):
    idx = list(range(len(X)))
    np.random.shuffle(idx)
    
    train_idx = idx[: int(len(X) * 0.7)]

    train_idxs.append(train_idx)


In [ ]:
# Генерациф случайных тестовых выборок
idx = list(range(len(X)))
np.random.shuffle(idx)

test_idx = idx[: int(len(X) * 0.3)]


In [ ]:
# Обучение моделей на всех подвыборках
pred = {}
num_epochs = 20

for d, model in models.items():
    
    pred[d] = []
    y_test = Y[test_idx]
    X_test = torch.Tensor(len(test_idx), d)
    
    for i in range(d):
        X_test[:, i] = X[test_idx] ** (i + 1)
        
    for train_idx in train_idxs:
        
        optimizer = optim.Adam(model.parameters(), lr = 0.01)
        
        X_train = torch.Tensor(len(train_idx), d)
        y_train = Y[train_idx]
        
        for i in range(d):
            X_test[:, i] = X[test_idx] ** (i + 1)
            
        for train_idx in train_idxs:
            
            optimizer = optim.Adam(model.parameters(), lr=0.01)
            
            X_train = torch.Tensor(len(train_idx), d)
            y_train = Y[train_idx]
            
            for i in range(d):
                X_train[:, i] = X[train_idx] ** (i + 1)
                
            # model train loop
            for epoch in range(num_epochs):
                
                optimizer.zero_grad()
                
                y_pred = model(X_train).squeeze()
                error = loss(y_pred, y_train)
                
                error.backward()
                optimizer.step()
                
            error = loss(model(X_test).squeeze(), y_test)
            pred[d].append(model(X_test).squeeze().detach().numpy())
        
    
    


In [ ]:
# Визцуализация обученных моделей
from matplotlib import cm

fig = plt.figure()
ax = fig.add_subplot(111)

for i, (d, model) in enumerate(models.items()):
    color = cm.viridis_r.colors[i * len(cm.viridis_r.colors) // len(models)]
    
    
    X_test = torch.Tensor(len(test_idx), d)
    
    for i in range(d):
        X_test[:, i] = X[test_idx] ** (i + 1)
        
    y_pred = model(X_test).squeeze()
    ax.scatter(X[test_idx], y_pred.tolist(), color=color, marker=".", label=f"d = {d}", alpha=0.5)
    
    
ax.set_xlabel('x')
ax.scatter(X[test_idx], Y[test_idx].tolist(), c='r', marker='.', label="f(x)")
ax.plot(X, g(X), c='gray', marker='.', label="g(X)")
plt.legend()


In [ ]:
# Вычисление смещения и дисперсии по обучающим выборкам

In [ ]:
bias_sq = {
    i : np.mean((g(X[test_idx]).numpy() - np.mean(pred[i], axis=0)) ** 2)
    for i in models.keys()
}

var = {
    i : np.array(pred[i]).var(axis=0).mean()
    for i in models.keys()
}


In [ ]:
from mmatplotlib import cm

fig = plt.figure()
ax = fig.add_subplot(111)

ax.set_xlabel('d')

ax.plot(list(bias_sq.keys()), list(bias_sq.values()), c='b', label="$bias^2$")
ax.plot(list(var.keys()), list(var.values()), c='g', label="Var")

plt.legend()